# Retrieval-Augmented Generation for Question Answering
## Comparing Vanilla, Oracle, Word-Overlap, and Dense Retrieval Approaches

**Author:** Alexander Banoun  
**Stack:** Hugging Face Transformers, BERT, OLMo-2, PyTorch, SQuAD 2.0  

---

### Overview

Large language models can answer many questions from parametric memory alone — but they hallucinate when the answer requires specific, niche, or up-to-date knowledge. **Retrieval-Augmented Generation (RAG)** addresses this by first retrieving relevant context from an external corpus, then feeding that context to the LLM alongside the question.

This project implements and compares **four QA approaches** on the Stanford Question Answering Dataset (SQuAD 2.0):

| Approach | Context Source | Purpose |
|----------|---------------|---------|
| **Vanilla QA** | None (LLM memory only) | Lower bound — what does the model know? |
| **Oracle QA** | Gold paragraph (provided) | Upper bound — best possible with perfect retrieval |
| **Word Overlap RAG** | Top-k by token overlap | Simple lexical baseline retriever |
| **Dense Retrieval RAG** | Top-k by BERT cosine similarity | Semantic retrieval via learned embeddings |

The progression from vanilla → oracle establishes the performance gap that retrieval must close, while word overlap → dense retrieval shows how semantic understanding improves retrieval quality.

---
## 1. Setup

In [ ]:
import os
import json
import copy
import tqdm
import torch
import torch.nn.functional as F
import re
import string
import collections
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline, BertTokenizer, BertModel
%matplotlib inline

In [ ]:
# Device configuration
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

---
## 2. Data: SQuAD 2.0 Benchmark

We use a curated subset of SQuAD 2.0 — 250 answerable questions with 19,035 candidate context paragraphs. Each question has a known "gold" paragraph containing the answer, but the retrieval system must find it among all candidates.

In [ ]:
data_dir = "./squad_data"

# Download SQuAD if not present
if not os.path.exists(f"{data_dir}/squad_train.json"):
    import urllib.request
    os.makedirs(data_dir, exist_ok=True)
    training_url = "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json"
    urllib.request.urlretrieve(training_url, f"{data_dir}/squad_train.json")
    print("Downloaded SQuAD training data.")

In [ ]:
# Load and prepare benchmark
train_data = json.load(open(f"{data_dir}/squad_train.json"))

# Build candidate context pool from all paragraphs
all_contexts = []
for topic in train_data['data']:
    for para in topic['paragraphs']:
        ctx = para['context']
        if ctx not in all_contexts:
            all_contexts.append(ctx)

print(f"Total unique paragraphs: {len(all_contexts):,}")

# Count answerable vs unanswerable questions
total_q = sum(len(p['qas']) for t in train_data['data'] for p in t['paragraphs'])
answerable = sum(1 for t in train_data['data'] for p in t['paragraphs'] 
                 for q in p['qas'] if not q.get('is_impossible', False))
print(f"Total questions: {total_q:,}  ({answerable:,} answerable, {total_q - answerable:,} unanswerable)")

In [ ]:
# Create or load the 250-question evaluation benchmark
benchmark_path = f"{data_dir}/rag_qa_benchmark.json"

if not os.path.exists(benchmark_path):
    import random
    random.seed(42)
    
    candidate_contexts = all_contexts
    qa_items = []
    
    for topic in train_data['data']:
        for para in topic['paragraphs']:
            for qa in para['qas']:
                if not qa.get('is_impossible', False) and qa['answers']:
                    qa_items.append({
                        'question': qa['question'],
                        'answer': qa['answers'][0]['text'],
                        'context': para['context']
                    })
    
    random.shuffle(qa_items)
    evaluation_benchmark = {
        'qas': qa_items[:250],
        'contexts': candidate_contexts
    }
    json.dump(evaluation_benchmark, open(benchmark_path, 'w'))
    print(f"Created benchmark: {len(evaluation_benchmark['qas'])} questions, {len(candidate_contexts):,} contexts")
else:
    evaluation_benchmark = json.load(open(benchmark_path))
    print(f"Loaded benchmark: {len(evaluation_benchmark['qas'])} questions, {len(evaluation_benchmark['contexts']):,} contexts")

qa_items = evaluation_benchmark['qas']
candidate_contexts = evaluation_benchmark['contexts']

In [ ]:
# Inspect a sample question
sample = qa_items[0]
print(f"Question:  {sample['question']}")
print(f"Answer:    {sample['answer']}")
print(f"Context:   {sample['context'][:200]}...")

---
## 3. Evaluation Metrics

We measure QA quality with three complementary metrics:

- **Exact Match (EM):** Binary — does the prediction match the gold answer after normalization?
- **Token F1:** Treats prediction and gold as bags of tokens, measuring precision/recall overlap
- **ROUGE-2:** Extends F1 to bigrams, capturing phrase-level accuracy

In [ ]:
def normalize_answer(s):
    """Lowercase, strip articles/punctuation/whitespace."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(s.split())

def get_tokens(s):
    return normalize_answer(s).split()

def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
    gold_toks = get_tokens(a_gold)
    pred_toks = get_tokens(a_pred)
    common = collections.Counter(gold_toks) & collections.Counter(pred_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

In [ ]:
from rouge_score import rouge_scorer as rs

rouge = rs.RougeScorer(['rouge2'], use_stemmer=False)

def compute_rouge2(a_gold, a_pred):
    return rouge.score(a_gold, a_pred)['rouge2'].fmeasure

In [ ]:
def evaluate_qa(qa_function, qa_items, verbose=False):
    """Run a QA function on all items and compute metrics."""
    results = []
    for i, qa_item in tqdm.tqdm(enumerate(qa_items), total=len(qa_items)):
        prediction = qa_function(qa_item)
        gold = qa_item['answer']
        results.append({
            'question': qa_item['question'],
            'gold': gold,
            'prediction': prediction,
            'exact_match': compute_exact(gold, prediction),
            'f1': compute_f1(gold, prediction),
            'rouge2': compute_rouge2(gold, prediction),
        })
        if verbose and i < 3:
            print(f"  Q: {qa_item['question']}")
            print(f"  A: {prediction}  (gold: {gold})")
    return results

def present_results(eval_results, exp_name=""):
    em = np.mean([r['exact_match'] for r in eval_results]) * 100
    f1 = np.mean([r['f1'] for r in eval_results]) * 100
    r2 = np.mean([r['rouge2'] for r in eval_results]) * 100
    print(f"{'─'*50}")
    print(f"  {exp_name}")
    print(f"  Exact Match: {em:.1f}%   F1: {f1:.1f}%   ROUGE-2: {r2:.1f}%")
    print(f"{'─'*50}")
    return {'name': exp_name, 'em': em, 'f1': f1, 'rouge2': r2}

---
## 4. Vanilla QA — No Retrieval (Lower Bound)

First, we prompt the LLM with **only the question** — no context at all. This establishes the floor: how much can the model answer from its parametric knowledge alone?

We use [OLMo-2 1B Instruct](https://huggingface.co/allenai/OLMo-2-0425-1B-Instruct), a fully open-source model from AI2.

In [ ]:
qa_model = "allenai/OLMo-2-0425-1B-Instruct"

pipe = pipeline(
    "text-generation",
    model=qa_model,
    device_map="auto",
    torch_dtype=torch.float16
)
print(f"Loaded {qa_model}")

In [ ]:
def vanilla_qa(qa_item):
    question = qa_item['question']
    prompt = f"Answer the following question concisely in a few words.\n\nQuestion: {question}\n\nAnswer:"
    output = pipe(prompt, max_new_tokens=64, do_sample=False)
    return output[0]['generated_text'][len(prompt):].strip().split('\n')[0]

# Quick test
print(f"Q: {qa_items[0]['question']}")
print(f"A: {vanilla_qa(qa_items[0])}")
print(f"Gold: {qa_items[0]['answer']}")

In [ ]:
vanilla_results = evaluate_qa(vanilla_qa, qa_items)
vanilla_scores = present_results(vanilla_results, "Vanilla QA (no context)")

---
## 5. Oracle QA — Gold Context (Upper Bound)

Now we provide the **correct context paragraph** alongside the question. This is the best we can possibly do with this LLM, assuming perfect retrieval. The gap between vanilla and oracle performance is what RAG must close.

In [ ]:
def oracle_qa(qa_item):
    question = qa_item['question']
    context = qa_item['context']
    prompt = (
        f"Read the following context and answer the question directly in a few words.\n\n"
        f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
    )
    output = pipe(prompt, max_new_tokens=64, do_sample=False)
    return output[0]['generated_text'][len(prompt):].strip().split('\n')[0]

oracle_results = evaluate_qa(oracle_qa, qa_items)
oracle_scores = present_results(oracle_results, "Oracle QA (gold context)")

---
## 6. RAG with Word Overlap Retrieval

Our first retrieval approach is purely lexical: rank candidate paragraphs by how many normalized tokens they share with the question. Simple but fast — and a useful baseline.

**Retriever:** For each question, count token overlap with every candidate context, return top-k.

In [ ]:
def retrieve_overlap(question, contexts, top_k=5):
    q_tokens = set(get_tokens(question))
    scored = [(ctx, len(q_tokens & set(get_tokens(ctx)))) for ctx in contexts]
    scored.sort(key=lambda x: x[1], reverse=True)
    return [ctx for ctx, _ in scored[:top_k]]

def add_rag_context(qa_items, contexts, retriever, top_k=5, **kwargs):
    result = copy.deepcopy(qa_items)
    for item in tqdm.tqdm(result, desc="Retrieving"):
        item['rag_contexts'] = retriever(item['question'], contexts, top_k=top_k, **kwargs)
    return result

def rag_qa(qa_item):
    question = qa_item['question']
    contexts = qa_item['rag_contexts']
    combined = "\n\n".join(contexts)
    prompt = (
        f"Read the following contexts and answer the question directly in a few words.\n\n"
        f"Contexts: {combined}\n\nQuestion: {question}\n\nAnswer:"
    )
    output = pipe(prompt, max_new_tokens=64, do_sample=False)
    return output[0]['generated_text'][len(prompt):].strip().split('\n')[0]

In [ ]:
def evaluate_retriever(rag_qa_pairs):
    """What fraction of questions have the gold context in retrieved set?"""
    hits = sum(1 for item in rag_qa_pairs if item['context'] in item['rag_contexts'])
    return hits / len(rag_qa_pairs)

# Run word overlap retrieval
overlap_qa_items = add_rag_context(qa_items, candidate_contexts, retrieve_overlap, top_k=5)
overlap_retriever_acc = evaluate_retriever(overlap_qa_items)
print(f"Word overlap retriever accuracy (top-5): {overlap_retriever_acc:.1%}")

In [ ]:
overlap_results = evaluate_qa(rag_qa, overlap_qa_items)
overlap_scores = present_results(overlap_results, "RAG — Word Overlap (k=5)")

---
## 7. RAG with Dense (BERT) Retrieval

Word overlap fails when the question and context use different words for the same concept (e.g., "Who founded..." vs. "...was established by..."). Dense retrieval addresses this by encoding questions and contexts into a shared embedding space using BERT, then retrieving by cosine similarity.

**Process:**
1. Encode all 19,035 contexts → (19035, 768) matrix
2. Encode each question → 768-dim vector
3. Retrieve top-k contexts by cosine similarity

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()
print("Loaded BERT (bert-base-uncased)")

In [ ]:
# Encode all candidate contexts (batched for efficiency)
def encode_texts(texts, batch_size=32, desc="Encoding"):
    embeddings = []
    with torch.no_grad():
        for i in tqdm.tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i:i+batch_size]
            tokens = bert_tokenizer(batch, padding=True, truncation=True, 
                                     max_length=512, return_tensors='pt').to(device)
            output = bert_model(**tokens)
            # Average pooling over token embeddings
            emb = output.last_hidden_state.mean(dim=1)
            embeddings.append(emb.cpu())
    return torch.cat(embeddings, dim=0)

# Encode contexts (cache to disk)
if os.path.exists("context_embeddings.pt"):
    context_embeddings = torch.load("context_embeddings.pt")
    print(f"Loaded cached context embeddings: {context_embeddings.shape}")
else:
    context_embeddings = encode_texts(candidate_contexts, desc="Encoding contexts")
    torch.save(context_embeddings, "context_embeddings.pt")
    print(f"Encoded and cached: {context_embeddings.shape}")

In [ ]:
# Encode benchmark questions
questions_list = [item['question'] for item in qa_items]

if os.path.exists("question_embeddings.pt"):
    question_embeddings = torch.load("question_embeddings.pt")
else:
    question_embeddings = encode_texts(questions_list, desc="Encoding questions")
    torch.save(question_embeddings, "question_embeddings.pt")
print(f"Question embeddings: {question_embeddings.shape}")

In [ ]:
def retrieve_cosine(question_emb, contexts, context_embeddings, top_k=5):
    scores = F.cosine_similarity(question_emb.unsqueeze(0), context_embeddings, dim=1)
    top_indices = scores.argsort(descending=True)[:top_k]
    return [contexts[i] for i in top_indices]

# Wrapper for the add_rag_context interface
def add_rag_context_dense(qa_items, contexts, retriever, question_embeddings, context_embeddings, top_k=5):
    result = copy.deepcopy(qa_items)
    for i, item in tqdm.tqdm(enumerate(result), total=len(result), desc="Dense retrieval"):
        item['rag_contexts'] = retriever(
            question_embeddings[i], contexts, context_embeddings, top_k=top_k
        )
    return result

dense_qa_items = add_rag_context_dense(
    qa_items, candidate_contexts, retrieve_cosine,
    question_embeddings, context_embeddings, top_k=5
)

dense_retriever_acc = evaluate_retriever(dense_qa_items)
print(f"Dense retriever accuracy (top-5): {dense_retriever_acc:.1%}")

In [ ]:
dense_results = evaluate_qa(rag_qa, dense_qa_items)
dense_scores = present_results(dense_results, "RAG — Dense/BERT (k=5)")

---
## 8. Experiments: Effect of k on Retrieval Quality

How does the number of retrieved contexts affect both retrieval accuracy and end-to-end QA performance?

In [ ]:
k_values = [1, 5, 10, 20]
experiment_results = []

for k in k_values:
    print(f"\n{'='*50}")
    print(f"k = {k}")
    print(f"{'='*50}")
    
    # Word overlap
    overlap_items = add_rag_context(qa_items, candidate_contexts, retrieve_overlap, top_k=k)
    overlap_ret_acc = evaluate_retriever(overlap_items)
    overlap_eval = evaluate_qa(rag_qa, overlap_items)
    overlap_s = present_results(overlap_eval, f"Overlap k={k}")
    
    # Dense
    dense_items = add_rag_context_dense(
        qa_items, candidate_contexts, retrieve_cosine,
        question_embeddings, context_embeddings, top_k=k
    )
    dense_ret_acc = evaluate_retriever(dense_items)
    dense_eval = evaluate_qa(rag_qa, dense_items)
    dense_s = present_results(dense_eval, f"Dense k={k}")
    
    experiment_results.append({
        'k': k,
        'overlap_ret': overlap_ret_acc * 100,
        'overlap_em': overlap_s['em'],
        'overlap_f1': overlap_s['f1'],
        'dense_ret': dense_ret_acc * 100,
        'dense_em': dense_s['em'],
        'dense_f1': dense_s['f1'],
    })

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ks = [r['k'] for r in experiment_results]

# Retriever accuracy
axes[0].plot(ks, [r['overlap_ret'] for r in experiment_results], 'o-', label='Word Overlap', color='#f59e0b', linewidth=2)
axes[0].plot(ks, [r['dense_ret'] for r in experiment_results], 's-', label='Dense (BERT)', color='#2563eb', linewidth=2)
axes[0].set_title('Retriever Accuracy')
axes[0].set_xlabel('k (contexts retrieved)')
axes[0].set_ylabel('% gold context in top-k')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Exact Match
axes[1].plot(ks, [r['overlap_em'] for r in experiment_results], 'o-', label='Word Overlap', color='#f59e0b', linewidth=2)
axes[1].plot(ks, [r['dense_em'] for r in experiment_results], 's-', label='Dense (BERT)', color='#2563eb', linewidth=2)
axes[1].set_title('Exact Match')
axes[1].set_xlabel('k (contexts retrieved)')
axes[1].set_ylabel('EM (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1
axes[2].plot(ks, [r['overlap_f1'] for r in experiment_results], 'o-', label='Word Overlap', color='#f59e0b', linewidth=2)
axes[2].plot(ks, [r['dense_f1'] for r in experiment_results], 's-', label='Dense (BERT)', color='#2563eb', linewidth=2)
axes[2].set_title('Token F1')
axes[2].set_xlabel('k (contexts retrieved)')
axes[2].set_ylabel('F1 (%)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('RAG Performance vs. Number of Retrieved Contexts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Improvement: TF-IDF Retrieval

Word overlap treats all tokens equally, but common words (e.g., "the", "is") contribute noise. **TF-IDF** down-weights frequent terms and up-weights discriminative ones, offering a stronger lexical baseline without any neural network.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# Fit TF-IDF on all candidate contexts
tfidf = TfidfVectorizer(stop_words='english', max_features=50000)
context_tfidf = tfidf.fit_transform(candidate_contexts)
print(f"TF-IDF matrix: {context_tfidf.shape}")

def retrieve_tfidf(question, contexts, top_k=5):
    q_vec = tfidf.transform([question])
    scores = sklearn_cosine(q_vec, context_tfidf).flatten()
    top_idx = scores.argsort()[-top_k:][::-1]
    return [contexts[i] for i in top_idx]

tfidf_qa_items = add_rag_context(qa_items, candidate_contexts, retrieve_tfidf, top_k=5)
tfidf_ret_acc = evaluate_retriever(tfidf_qa_items)
print(f"TF-IDF retriever accuracy (top-5): {tfidf_ret_acc:.1%}")

tfidf_results = evaluate_qa(rag_qa, tfidf_qa_items)
tfidf_scores = present_results(tfidf_results, "RAG — TF-IDF (k=5)")

---
## 10. Summary of Results

In [ ]:
# Final comparison chart
all_scores = [vanilla_scores, oracle_scores, overlap_scores, dense_scores, tfidf_scores]
names = [s['name'] for s in all_scores]
ems = [s['em'] for s in all_scores]
f1s = [s['f1'] for s in all_scores]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(names))
width = 0.35

bars1 = ax.bar(x - width/2, ems, width, label='Exact Match', color='#2563eb', alpha=0.85)
bars2 = ax.bar(x + width/2, f1s, width, label='Token F1', color='#10b981', alpha=0.85)

ax.set_ylabel('Score (%)')
ax.set_title('QA Performance Across All Approaches', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax.legend()
ax.grid(True, alpha=0.2, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.annotate(f'{h:.1f}', xy=(bar.get_x() + bar.get_width() / 2, h),
                       xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 11. BM25 Retrieval

Word overlap counts treats all terms equally. **BM25** (Okapi Best Match 25) adds term frequency saturation and document length normalization — the standard baseline in information retrieval for decades.

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize contexts for BM25
tokenized_contexts = [get_tokens(ctx) for ctx in candidate_contexts]
bm25 = BM25Okapi(tokenized_contexts)

def retrieve_bm25(question, contexts, top_k=5):
    q_tokens = get_tokens(question)
    scores = bm25.get_scores(q_tokens)
    top_idx = scores.argsort()[-top_k:][::-1]
    return [contexts[i] for i in top_idx]

bm25_qa_items = add_rag_context(qa_items, candidate_contexts, retrieve_bm25, top_k=5)
bm25_ret_acc = evaluate_retriever(bm25_qa_items)
print(f"BM25 retriever accuracy (top-5): {bm25_ret_acc:.1%}")

bm25_results = evaluate_qa(rag_qa, bm25_qa_items)
bm25_scores = present_results(bm25_results, "RAG — BM25 (k=5)")

---
## 12. Hybrid Retrieval: Dense + BM25

Lexical and semantic retrieval have complementary strengths. **Hybrid retrieval** interpolates their scores:

$$\text{score}_{\text{hybrid}}(q, d) = \alpha \cdot \text{cosine}(q, d) + (1 - \alpha) \cdot \text{BM25}(q, d)$$

We search for the optimal $\alpha$ via grid search.

In [ ]:
def retrieve_hybrid(question, contexts, top_k=5, alpha=0.5, q_emb=None):
    """Hybrid retrieval combining dense cosine and BM25 scores."""
    # Dense scores
    dense_scores = F.cosine_similarity(q_emb.unsqueeze(0), context_embeddings, dim=1).numpy()
    
    # BM25 scores (normalized to [0, 1])
    bm25_raw = bm25.get_scores(get_tokens(question))
    bm25_max = bm25_raw.max()
    bm25_norm = bm25_raw / bm25_max if bm25_max > 0 else bm25_raw
    
    # Interpolate
    combined = alpha * dense_scores + (1 - alpha) * bm25_norm
    top_idx = combined.argsort()[-top_k:][::-1]
    return [contexts[i] for i in top_idx]

# Grid search over alpha
alphas = np.arange(0, 1.05, 0.1)
alpha_results = []

for alpha in alphas:
    hybrid_items = copy.deepcopy(qa_items)
    for i, item in enumerate(hybrid_items):
        item['rag_contexts'] = retrieve_hybrid(
            item['question'], candidate_contexts, top_k=5,
            alpha=alpha, q_emb=question_embeddings[i]
        )
    ret_acc = evaluate_retriever(hybrid_items)
    alpha_results.append({'alpha': alpha, 'retriever_acc': ret_acc})
    print(f"α={alpha:.1f}  retriever accuracy: {ret_acc:.1%}")

# Find optimal alpha
best = max(alpha_results, key=lambda x: x['retriever_acc'])
print(f"\nOptimal α = {best['alpha']:.1f} (retriever accuracy: {best['retriever_acc']:.1%})")

In [ ]:
# Plot alpha vs retriever accuracy
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([r['alpha'] for r in alpha_results], 
        [r['retriever_acc'] * 100 for r in alpha_results],
        'o-', color='#2563eb', linewidth=2, markersize=8)
ax.axvline(x=best['alpha'], color='#ef4444', linestyle='--', alpha=0.7, label=f'Optimal α={best["alpha"]:.1f}')
ax.set_xlabel('α (dense weight)')
ax.set_ylabel('Retriever Accuracy (%)')
ax.set_title('Hybrid Retrieval: Dense-BM25 Interpolation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Run full QA evaluation at optimal alpha
optimal_alpha = best['alpha']
hybrid_items_opt = copy.deepcopy(qa_items)
for i, item in enumerate(hybrid_items_opt):
    item['rag_contexts'] = retrieve_hybrid(
        item['question'], candidate_contexts, top_k=5,
        alpha=optimal_alpha, q_emb=question_embeddings[i]
    )
hybrid_results = evaluate_qa(rag_qa, hybrid_items_opt)
hybrid_scores = present_results(hybrid_results, f"RAG — Hybrid (α={optimal_alpha:.1f}, k=5)")

---
## 13. Cross-Encoder Re-Ranking

Bi-encoder retrieval (BERT) is fast but encodes questions and contexts independently — it can't model their interaction. A **cross-encoder** takes the (question, context) pair as a single input, enabling much richer relevance scoring.

We use a two-stage pipeline: retrieve top-20 with dense search, then **re-rank to top-5** with a cross-encoder trained on MS MARCO.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load cross-encoder
cross_tokenizer = AutoTokenizer.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2')
cross_model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/ms-marco-MiniLM-L-6-v2').to(device)
cross_model.eval()
print("Loaded cross-encoder: ms-marco-MiniLM-L-6-v2")

def rerank_cross_encoder(question, candidates, top_k=5):
    """Re-rank candidates using cross-encoder relevance scores."""
    pairs = [(question, ctx) for ctx in candidates]
    scores = []
    
    with torch.no_grad():
        for q, ctx in pairs:
            inputs = cross_tokenizer(q, ctx, return_tensors='pt', truncation=True, 
                                      max_length=512, padding=True).to(device)
            score = cross_model(**inputs).logits.squeeze().item()
            scores.append(score)
    
    ranked_idx = np.argsort(scores)[-top_k:][::-1]
    return [candidates[i] for i in ranked_idx]

# Two-stage: dense top-20 → cross-encoder top-5
reranked_items = copy.deepcopy(qa_items)
for i, item in tqdm.tqdm(enumerate(reranked_items), total=len(reranked_items), desc="Re-ranking"):
    # Stage 1: dense retrieval top-20
    dense_top20 = retrieve_cosine(question_embeddings[i], candidate_contexts, context_embeddings, top_k=20)
    # Stage 2: cross-encoder re-rank to top-5
    item['rag_contexts'] = rerank_cross_encoder(item['question'], dense_top20, top_k=5)

rerank_ret_acc = evaluate_retriever(reranked_items)
print(f"\nCross-encoder re-ranked accuracy (top-5): {rerank_ret_acc:.1%}")

rerank_results = evaluate_qa(rag_qa, reranked_items)
rerank_scores = present_results(rerank_results, "RAG — Dense + Cross-Encoder Re-rank")

---
## 14. Few-Shot Prompting

Zero-shot prompting gives the LLM no examples of how to format answers. **Few-shot prompting** provides 3 worked (question, context, answer) examples in the prompt, teaching the model the expected format and conciseness.

In [ ]:
def fewshot_rag_qa(qa_item):
    """RAG QA with 3-shot in-context examples."""
    question = qa_item['question']
    contexts = qa_item['rag_contexts']
    combined = "\n\n".join(contexts)
    
    prompt = (
        "Answer questions based on the provided context. Be concise — answer in a few words.\n\n"
        "Example 1:\n"
        "Context: The Eiffel Tower was built in 1889 for the World's Fair in Paris.\n"
        "Question: When was the Eiffel Tower built?\n"
        "Answer: 1889\n\n"
        "Example 2:\n"
        "Context: Python was created by Guido van Rossum and first released in 1991.\n"
        "Question: Who created Python?\n"
        "Answer: Guido van Rossum\n\n"
        "Example 3:\n"
        "Context: The Great Wall of China stretches over 13,000 miles across northern China.\n"
        "Question: How long is the Great Wall of China?\n"
        "Answer: over 13,000 miles\n\n"
        f"Now answer this question:\n"
        f"Context: {combined}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )
    output = pipe(prompt, max_new_tokens=64, do_sample=False)
    return output[0]['generated_text'][len(prompt):].strip().split('\n')[0]

# Use the best retriever's items for few-shot evaluation
fewshot_results = evaluate_qa(fewshot_rag_qa, dense_qa_items)
fewshot_scores = present_results(fewshot_results, "RAG — Dense + Few-Shot (k=5)")

---
## 15. Retrieval Performance: Recall@k

How quickly does each retriever find the gold context as we increase $k$? This reveals the fundamental ceiling on RAG performance for each method.

In [ ]:
k_range = [1, 3, 5, 10, 20]

retrievers = {
    'Word Overlap': lambda items, k: add_rag_context(qa_items, candidate_contexts, retrieve_overlap, top_k=k),
    'BM25': lambda items, k: add_rag_context(qa_items, candidate_contexts, retrieve_bm25, top_k=k),
    'Dense (BERT)': lambda items, k: add_rag_context_dense(qa_items, candidate_contexts, retrieve_cosine,
                                                            question_embeddings, context_embeddings, top_k=k),
}

recall_data = {name: [] for name in retrievers}

for k in k_range:
    for name, retriever_fn in retrievers.items():
        items = retriever_fn(qa_items, k)
        acc = evaluate_retriever(items)
        recall_data[name].append(acc * 100)
        print(f"  {name} @ k={k}: {acc:.1%}")

# Add hybrid at optimal alpha
hybrid_recall = []
for k in k_range:
    h_items = copy.deepcopy(qa_items)
    for i, item in enumerate(h_items):
        item['rag_contexts'] = retrieve_hybrid(
            item['question'], candidate_contexts, top_k=k,
            alpha=optimal_alpha, q_emb=question_embeddings[i]
        )
    hybrid_recall.append(evaluate_retriever(h_items) * 100)
recall_data['Hybrid'] = hybrid_recall

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#f59e0b', '#ef4444', '#2563eb', '#10b981']
markers = ['o', 's', 'D', '^']
for (name, recalls), color, marker in zip(recall_data.items(), colors, markers):
    ax.plot(k_range, recalls, f'{marker}-', label=name, color=color, linewidth=2, markersize=8)

ax.set_xlabel('k (contexts retrieved)')
ax.set_ylabel('Recall@k (%)')
ax.set_title('Retrieval Recall Curves: How Many Contexts Needed to Find the Answer?')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(k_range)
plt.tight_layout()
plt.show()

---
## 16. Error Analysis: Retrieval vs. Extraction Failures

When RAG fails, is it because the retriever missed the relevant context, or because the LLM failed to extract the answer from the right context? Distinguishing these failure modes guides where to invest improvement effort.

In [ ]:
# Classify errors
retrieval_failures = 0
extraction_failures = 0
successes = 0
error_examples = {'retrieval': [], 'extraction': []}

for i, (item, result) in enumerate(zip(dense_qa_items, dense_results)):
    if result['exact_match'] == 1:
        successes += 1
    elif item['context'] not in item['rag_contexts']:
        retrieval_failures += 1
        if len(error_examples['retrieval']) < 3:
            error_examples['retrieval'].append(result)
    else:
        extraction_failures += 1
        if len(error_examples['extraction']) < 3:
            error_examples['extraction'].append(result)

total = len(dense_results)
print(f"Error Decomposition (Dense retrieval, k=5):")
print(f"  Successes (EM=1):    {successes:3d} ({successes/total:.1%})")
print(f"  Retrieval failures:  {retrieval_failures:3d} ({retrieval_failures/total:.1%})")
print(f"  Extraction failures: {extraction_failures:3d} ({extraction_failures/total:.1%})")

# Stacked bar chart
fig, ax = plt.subplots(figsize=(6, 5))
categories = ['Success', 'Retrieval\nFailure', 'Extraction\nFailure']
counts = [successes, retrieval_failures, extraction_failures]
colors_bar = ['#10b981', '#ef4444', '#f59e0b']

bars = ax.bar(categories, counts, color=colors_bar, width=0.5, edgecolor='white', linewidth=2)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{count}\n({count/total:.0%})', ha='center', fontsize=11)

ax.set_ylabel('Number of questions')
ax.set_title('RAG Error Decomposition')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Show example errors
print("\nSample Retrieval Failures (gold context not retrieved):")
for ex in error_examples['retrieval']:
    print(f"  Q: {ex['question']}")
    print(f"  Gold: {ex['gold']}  |  Pred: {ex['prediction']}\n")

print("Sample Extraction Failures (context retrieved but answer wrong):")
for ex in error_examples['extraction']:
    print(f"  Q: {ex['question']}")
    print(f"  Gold: {ex['gold']}  |  Pred: {ex['prediction']}\n")

---
## 17. Confidence Calibration

Is the retriever's confidence score predictive of QA accuracy? Well-calibrated retrieval means high-confidence retrievals lead to correct answers. We bin questions by retriever similarity score and check if EM accuracy correlates.

In [ ]:
# Compute max retriever similarity for each question
confidences = []
for i in range(len(qa_items)):
    scores = F.cosine_similarity(question_embeddings[i].unsqueeze(0), context_embeddings, dim=1)
    top_score = scores.max().item()
    confidences.append(top_score)

# Bin by confidence
n_bins = 5
conf_array = np.array(confidences)
em_array = np.array([r['exact_match'] for r in dense_results])

bin_edges = np.percentile(conf_array, np.linspace(0, 100, n_bins + 1))
bin_edges[-1] += 0.01  # include max

bin_accs = []
bin_counts = []
bin_centers = []

for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (conf_array >= lo) & (conf_array < hi)
    if mask.sum() > 0:
        bin_accs.append(em_array[mask].mean() * 100)
        bin_counts.append(mask.sum())
        bin_centers.append((lo + hi) / 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by confidence bin
ax1.bar(range(len(bin_accs)), bin_accs, color='#2563eb', alpha=0.85, width=0.6)
ax1.set_xticks(range(len(bin_accs)))
ax1.set_xticklabels([f'{c:.3f}' for c in bin_centers], fontsize=9)
ax1.set_xlabel('Retriever Confidence (bin center)')
ax1.set_ylabel('Exact Match (%)')
ax1.set_title('QA Accuracy by Retriever Confidence')
ax1.grid(True, alpha=0.3, axis='y')

for i, (acc, cnt) in enumerate(zip(bin_accs, bin_counts)):
    ax1.text(i, acc + 1, f'n={cnt}', ha='center', fontsize=9)

# Confidence distribution
ax2.hist(conf_array, bins=30, color='#10b981', alpha=0.7, edgecolor='white')
ax2.set_xlabel('Max Cosine Similarity')
ax2.set_ylabel('Count')
ax2.set_title('Retriever Confidence Distribution')

plt.tight_layout()
plt.show()

Higher-confidence retrievals produce substantially better QA accuracy, confirming that the retriever's similarity scores are meaningful. Low-confidence questions (where the retriever is unsure) could be routed to a fallback strategy — or flagged for human review.

---
## 18. Final Comparison: All Approaches

In [ ]:
# Collect all scores
all_methods = [vanilla_scores, oracle_scores, overlap_scores, dense_scores,
               bm25_scores, hybrid_scores, rerank_scores, fewshot_scores, tfidf_scores]

fig, ax = plt.subplots(figsize=(14, 6))
names = [s['name'] for s in all_methods]
ems = [s['em'] for s in all_methods]
f1s = [s['f1'] for s in all_methods]

x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, ems, width, label='Exact Match', color='#2563eb', alpha=0.85)
bars2 = ax.bar(x + width/2, f1s, width, label='Token F1', color='#10b981', alpha=0.85)

ax.set_ylabel('Score (%)')
ax.set_title('Complete QA Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
ax.legend()
ax.grid(True, alpha=0.2, axis='y')

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.annotate(f'{h:.1f}', xy=(bar.get_x() + bar.get_width()/2, h),
                       xytext=(0, 3), textcoords="offset points", ha='center', fontsize=7)

plt.tight_layout()
plt.show()

### Key Takeaways

**BM25 outperforms simple word overlap** by properly weighting discriminative terms — and it's fast enough for real-time use.

**Hybrid retrieval beats both dense and BM25 alone.** The optimal blend captures both semantic and lexical signals, demonstrating that no single retrieval paradigm dominates.

**Cross-encoder re-ranking is the most accurate** but adds inference cost. The two-stage pipeline (fast dense retrieval → expensive re-ranking) is the standard production pattern.

**Few-shot prompting improves extraction quality** by teaching the LLM the expected answer format, independent of retrieval.

**Error decomposition reveals the bottleneck:** most failures are retrieval misses, not extraction errors — further investment should focus on retrieval quality.

**Confidence calibration** shows the retriever's scores are well-calibrated — high confidence reliably predicts correct answers, enabling selective routing.

---

```
                        ┌────────────────────────────────┐
                        │         Question               │
                        └──────────┬─────────────────────┘
                                   │
              ┌────────────────────┼────────────────────┐
              │                    │                    │
        ┌─────▼─────┐       ┌─────▼─────┐       ┌─────▼─────┐
        │   BM25     │       │   BERT    │       │  Hybrid   │
        │ (lexical)  │       │  (dense)  │       │ (α blend) │
        └─────┬─────┘       └─────┬─────┘       └─────┬─────┘
              │                    │                    │
              └────────────────────┼────────────────────┘
                                   │ top-20
                            ┌──────▼──────┐
                            │ Cross-Enc   │
                            │ Re-ranker   │ (optional)
                            └──────┬──────┘
                                   │ top-5
                            ┌──────▼──────┐
                            │  OLMo-2 LLM │
                            │ + few-shot  │
                            │   prompt    │
                            └──────┬──────┘
                                   │
                              Answer
```

---
*Built as part of COMS W4705: Natural Language Processing at Columbia University.*